# Recipe — experimental design via capacity-floor predictor

**One question:** I'm planning a perturbation experiment on cell type X to engage pathway Y. Before I run it, will Y actually ramp up, or is it already saturated in cell type X?

**No user data needed** — the bundled HPA reference (154 cell types × 44 modules) answers this directly via the upward-asymmetric capacity-floor predictor.

**Use this for:**
- Drug treatment design — pick the cell line where your target pathway has ramp room, not the one where it's already maxed
- Differentiation / commitment experiments — predict which architecture modules are saturated at the starting cell type
- Avoiding wasted experiments — a pre-experiment check that costs 30 seconds and saves wet-lab time

In [1]:
import perceptome as pct
import pandas as pd

# Show the bundled HPA reference dimensions
ref = pct.load_hpa_perceptivity()
print(f"HPA reference: {ref['R'].shape[0]} cell types × {ref['R'].shape[1]} modules")
print(f"Sample cell types: {list(ref['R'].index[:5])} ...")
print(f"Sample modules: {list(ref['R'].columns[:5])} ...")

HPA reference: 154 cell types × 44 modules
Sample cell types: ['adipocytes', 'adrenal cortex cells', 'adrenal medulla cells', 'alveolar cells type 1', 'alveolar cells type 2'] ...
Sample modules: ['AMPK', 'AR', 'AhR', 'Autophagy', 'BMP'] ...


## Single-cell-type query

**Scenario:** I'm planning a hepatocyte regeneration experiment and want to engage UPR-ATF6, HSF1, and mTOR. Which of these will ramp up?

In [2]:
pred = pct.predict_engagement(
    starting_cell_type="hepatocytes",
    operation_modules=["UPR-ATF6", "HSF1", "mTOR", "NRF2", "Autophagy"],
)
print(pred[["A_baseline", "headroom", "capacity_floor", "predicted_direction"]].to_string())

           A_baseline  headroom        capacity_floor       predicted_direction
module                                                                         
UPR-ATF6     6.316189  0.963517  saturated_blocked_up  up_blocked_down_possible
HSF1         2.665918  2.234511          intermediate    intermediate_uncertain
mTOR         4.880333  0.636630  saturated_blocked_up  up_blocked_down_possible
NRF2         4.422901  1.441615          intermediate    intermediate_uncertain
Autophagy    4.554198  0.910896  saturated_blocked_up  up_blocked_down_possible


**Reading the result:**

- **`saturated_blocked_up`** = HPA shows this module is already at saturation in this cell type. **Cannot ramp UP** under stimulus. (Active suppression DOWN by specific signaling — e.g. retinoic acid → UPR — IS still possible.)
- **`capacious`** = pathway has ramp room. Magnitude of ramp depends on how strongly your operation engages it (factor 2 of the two-factor framework, not in the tool).
- **`intermediate`** = uncertain — depends on operation intensity.

For the hepatocyte case above: UPR-ATF6 is at A_baseline ≈ 6.3 → blocked up. mTOR similar. **Conclusion**: don't expect strong UP-ramps on these for partial-hepatectomy regeneration; the pathways are already engaged at homeostatic baseline. Pick HSF1 or NRF2 if you need a clear positive signal.

## Cross-cell-type comparison — pick the right model system

**Scenario:** I want to study HSF1 induction. Which cell type gives me the cleanest readout (most ramp room from low baseline)?

In [3]:
# Loop over a panel of common research cell types
panel = [
    "cardiomyocytes", "hepatocytes", "alveolar cells type 2",
    "fibroblasts", "skeletal myocytes", "brain excitatory neurons",
    "b-cells", "t-cells", "macrophages",
]
rows = []
for ct in panel:
    if ct not in ref["R"].index:
        # Try fuzzy match
        cands = [c for c in ref["R"].index if ct.lower() in c.lower()]
        if not cands:
            continue
        ct = cands[0]
    pred = pct.predict_engagement(ct, ["HSF1"])
    rows.append({
        "cell_type": ct,
        "A_baseline": pred.loc["HSF1", "A_baseline"],
        "headroom": pred.loc["HSF1", "headroom"],
        "capacity_floor": pred.loc["HSF1", "capacity_floor"],
    })
panel_df = pd.DataFrame(rows).sort_values("headroom", ascending=False)
print("HSF1 capacity panel (highest headroom = best ramp candidate):")
print(panel_df.to_string(index=False))

HSF1 capacity panel (highest headroom = best ramp candidate):
               cell_type  A_baseline  headroom       capacity_floor
          cardiomyocytes    2.032003  2.868427            capacious
brain excitatory neurons    2.383015  2.517415            capacious
             hepatocytes    2.665918  2.234511         intermediate
   alveolar cells type 2    3.523315  1.377115         intermediate
                 b-cells    3.836394  1.064036         intermediate
             fibroblasts    4.020253  0.880177         intermediate
                 t-cells    4.222968  0.677462         intermediate
             macrophages    4.779182  0.121248 saturated_blocked_up


**Reading the panel:**

- Cells at the top of the headroom column are the best HSF1-engagement candidates — they have low baseline and lots of room to ramp UP under heat shock or proteostatic stress.
- Cells with `saturated_blocked_up` are at the ceiling — picking them means you'll see no signal even with a real biological effect.
- This was used a-priori in Paper 4.4: cardiomyocyte HSF1 was predicted `capacious` (A_baseline ≈ 2.0) → muscle hypertrophy experiment confirmed strong ramp d=+1.29. ✓

## Reference

- The capacity-floor predictor was **closed across paper4.5 + 4.6 + 4.7 + 4.8** (intestinal homeostasis, organoid differentiation, organoid regeneration, organoid retinoid perturbation) as upward-asymmetric.
- Threshold values (`A_baseline > 4.5` = saturated, `< 2.5` = capacious) come from validation across 4 datasets in 3 paradigms.
- For the full scope + paper4.8 PARTIAL refinement (specific signaling can DOWN-suppress saturated modules), see [`docs/SCOPE.md`](../docs/SCOPE.md).